<img src='https://hilpisch.com/taim_logo.png' width="350px" align="right">

# Certificate Programs

**Mentoring Session**

# Reinforcement Learning

**From Playing Games to Intraday Algorithmic Trading**

Dr Yves J Hilpisch | The AI Machine

http://aimachine.io | http://twitter.com/dyjh

## General Imports

In [ ]:
!git clone https://github.com/tpq-classes/python_for_algo_trading_addon.git
import sys
sys.path.append('python_for_algo_trading_addon')


In [ ]:
import os
import math
import time
import random
import numpy as np
import pandas as pd
from pylab import plt
from pyvirtualdisplay import Display
display = Display(visible=0, size=(1400, 900))
display.start()
from IPython import display
plt.ion()
plt.style.use('seaborn-v0_8')
%matplotlib inline
np.set_printoptions(precision=4, suppress=True)
os.environ['PYTHONHASHSEED'] = '0'

## OpenAI Environment

In [ ]:
import gym

In [ ]:
env = gym.make('CartPole-v0')

In [ ]:
env.observation_space

In [ ]:
env.observation_space.low.astype(np.float16)

In [ ]:
env.observation_space.high.astype(np.float16)

In [ ]:
state = env.reset()

In [ ]:
state # [cart position, cart velocity, pole angle, pole angular velocity]

In [ ]:
env.action_space

In [ ]:
env.action_space.n

In [ ]:
env.action_space.sample()

In [ ]:
env.action_space.sample() 

In [ ]:
a = env.action_space.sample()
a

In [ ]:
state, reward, done, info = env.step(a)
state, reward, done, info

In [ ]:
env.reset()
for e in range(1, 200):
    a = env.action_space.sample()
    state, reward, done, info = env.step(a)
    print(f'step={e:2d} | state={state} | action={a} | reward={reward}')
    if done and (e + 1) < 200:
        print('*** FAILED ***')
        break

In [ ]:
done

In [ ]:
env.reset()
img = plt.imshow(env.render(mode='rgb_array')) # initialize bitmap embedding
for e in range(200):
    img.set_data(env.render(mode='rgb_array')) # updating the data
    display.display(plt.gcf())
    display.clear_output(wait=True)
    a = env.action_space.sample()  # random action choice
    obs, rew, done, _ = env.step(a)  # taking action
    if done and (e + 1) < 200:
        print('*** FAILED ***')
        break

## Keras and Seeds

In [ ]:
import keras
import tensorflow as tf
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import accuracy_score

https://github.com/tensorflow/tensorflow/issues/33024

In [ ]:
tf.compat.v1.disable_eager_execution()

In [ ]:
tf.__version__

In [ ]:
keras.__version__

In [ ]:
def set_seeds(seed=100):
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)
    env.seed(seed)

## DQL Agent

See https://keon.io/deep-q-learning/

In [ ]:
from collections import deque

In [ ]:
class DQLAgent:
    def __init__(self, gamma=0.95, lr=0.001, finish=1e10):
        self.finish = finish
        self.epsilon = 1.0
        self.epsilon_min = 0.01
        self.epsilon_decay = 0.995
        self.gamma = gamma
        self.batch_size = 32
        self.lr = lr
        self.max_treward = 0
        self.averages = list()
        self.performances = list()  # only for trading
        self.aperformances = list()  # only for trading
        self.memory = deque(maxlen=2000)
        self.osn = env.observation_space.shape[0]
        self.model = self._build_model()
        
    def _build_model(self):
        model = Sequential()
        model.add(Dense(24, input_dim=self.osn,
                        activation='relu'))
        model.add(Dense(24, activation='relu'))
        model.add(Dense(env.action_space.n, activation='linear'))
        model.compile(loss='mse', optimizer=Adam(lr=self.lr))
        return model
        
    def act(self, state):
        if random.random() <= self.epsilon:
            return env.action_space.sample()
        action = self.model.predict(state)[0]
        return np.argmax(action)
    
    def replay(self):
        batch = random.sample(self.memory, self.batch_size)
        for state, action, reward, next_state, done in batch:
            if not done:
                reward += self.gamma * np.amax(
                    self.model.predict(next_state)[0])
            target = self.model.predict(state)
            target[0, action] = reward
            self.model.fit(state, target, epochs=1,
                           verbose=False)
        if self.epsilon > self.epsilon_min:
            self.epsilon *= self.epsilon_decay
    
    def learn(self, episodes, max_iter=200):
        trewards = []
        av = 0
        for e in range(1, episodes + 1):
            state = env.reset()
            state = np.reshape(state, [1, self.osn])
            for _ in range(max_iter):
                action = self.act(state)
                next_state, reward, done, info = env.step(action)
                next_state = np.reshape(next_state,
                                        [1, self.osn])
                self.memory.append([state, action, reward,
                                     next_state, done])
                state = next_state
                if done:
                    treward = _ + 1
                    trewards.append(treward)
                    av = sum(trewards[-25:]) / 25
                    self.averages.append(av)
                    self.max_treward = max(self.max_treward, treward)
                    templ = 'episode: {:4d}/{} | treward: {:4d} | '
                    templ += 'av: {:6.1f} | max: {:4d}'
                    print(templ.format(e, episodes, treward, av,
                                       self.max_treward), end='\r')
                    try:
                        # only for trading
                        self.performances.append(env.performance)
                        self.aperformances.append(sum(self.performances[-25:]) / 25)
                    except:
                        pass
                    break
            if av > self.finish:
                break
            if len(self.memory) > self.batch_size:
                self.replay()
    def test(self, episodes, max_iter=200):
        trewards = []
        performances = []
        for e in range(1, episodes + 1):
            state = env.reset()
            for _ in range(max_iter):
                state = np.reshape(state, [1, self.osn])
                action = np.argmax(self.model.predict(state)[0])
                next_state, reward, done, info = env.step(action)
                state = next_state
                if done:
                    treward = _ + 1
                    trewards.append(treward)
                    print('episode: {:4d}/{} | treward: {:4d}'
                          .format(e, episodes, treward), end='\r')
                    try:
                        performances.append(env.performance)
                    except:
                        pass
                    time.sleep(0.05)
                    break
        return trewards, performances

In [ ]:
set_seeds(100)
agent = DQLAgent(lr=0.001, finish=195)

In [ ]:
episodes = 1000

In [ ]:
%time agent.learn(episodes)

In [ ]:
agent.epsilon

In [ ]:
plt.figure(figsize=(10, 6))
x = range(len(agent.averages))
y = np.polyval(np.polyfit(x, agent.averages, deg=3), x)
plt.plot(agent.averages, label='moving average')
plt.plot(x, y, 'r--', label='regression')
plt.xlabel('episodes')
plt.ylabel('total reward')
plt.legend();

In [ ]:
trewards, _ = agent.test(100)

In [ ]:
sum(trewards) / len(trewards)

## Oanda Environment

In [ ]:
import tpqoa

In [ ]:
class observation_space:
    def __init__(self, n):
        self.shape = (n,)

In [ ]:
class action_space:
    def __init__(self, n):
        self.n = n
    def sample(self):
        return random.randint(0, self.n - 1)

In [ ]:
class OandaEnv:
    def __init__(self, symbol, start, end, granularity, price, feature='r',
                 leverage=50, min_accuracy=0.5, min_performance=0.85):
        self.symbol = symbol
        self.start = start
        self.end = end
        self.granularity = granularity
        self.price = price
        self.feature = feature
        self.lags = 4
        self.api = tpqoa.tpqoa('../../../pyalgo.cfg')
        self.leverage = leverage
        self.min_accuracy = min_accuracy
        self.min_performance = min_performance
        self.observation_space = observation_space(self.lags)
        self.action_space = action_space(2)
        self._get_data()
        self._prepare_data()

    def _get_data(self):
        ''' Method to retrieve data from Oanda.
        '''
        self.raw = self.api.get_history(self.symbol, self.start,
                                       self.end, self.granularity,
                                       self.price)
        self.data = pd.DataFrame(self.raw['c'])
        self.data.columns = [self.symbol]

    def _prepare_data(self):
        ''' Method to prepare additional time series data
            (such as features data).
        '''
        self.data['r'] = np.log(self.data / self.data.shift(1))
        self.data.dropna(inplace=True)
        self.mu = self.data.mean()
        self.std = self.data.std()
        self.data_ = (self.data - self.mu) / self.std
        self.data['d'] = np.where(self.data['r'] > 0, 1, 0)
        self.data['d'] = self.data['d'].astype(int)

    def _get_state(self):
        ''' Privat method that returns the state of the environment.
        '''
        return self.data_[self.feature].iloc[self.bar -
                                    self.lags:self.bar].values
    def seed(self, seed):
        pass
    def get_state(self, bar):
        ''' Method that returns the state of the environment.
        '''
        return self.data_[self.feature].iloc[bar - self.lags:bar].values

    def reset(self):
        ''' Method to reset the environment.
        '''
        self.treward = 0
        self.accuracy = 0
        self.performance = 1
        self.bar = self.lags
        state = self._get_state()
        return state

    def step(self, action):
        ''' Method to step the environment forwards.
        '''
        correct = action == self.data['d'].iloc[self.bar]
        ret = self.data['r'].iloc[self.bar] * self.leverage
        reward_1 = 1 if correct else 0
        reward_2 = abs(ret) if correct else -abs(ret)
        reward = reward_1 + reward_2 * self.leverage
        self.treward += reward_1
        self.bar += 1
        self.accuracy = self.treward / (self.bar - self.lags)
        self.performance *= math.exp(reward_2)
        if self.bar >= len(self.data):
            done = True
        elif reward_1 == 1:
            done = False
        elif (self.accuracy < self.min_accuracy and
              self.bar > self.lags + 15):
            done = True
        elif (self.performance < self.min_performance and
              self.bar > self.lags + 15):
            done = True
        else:
            done = False
        state = self._get_state()
        info = {}
        return state, reward, done, info

In [ ]:
symbol = 'EUR_USD'

In [ ]:
date = '2020-06-26'

In [ ]:
%%time
env = OandaEnv(symbol=symbol,
                  start=f'{date} 08:00:00',
                  end=f'{date} 11:00:00',
                  granularity='S5',
                  price='M',
                  feature=['r'],
                  min_accuracy=0.4,
                  min_performance=0.85
                 )

In [ ]:
env.data.info()

In [ ]:
env.reset()

In [ ]:
a = env.action_space.sample()
a

In [ ]:
env.step(a)

## Trading Bot

In [ ]:
set_seeds(100)
agent = DQLAgent(lr=0.001, gamma=0.5, finish=1550)

In [ ]:
episodes = 300

In [ ]:
agent.learn(episodes, max_iter=2100)

In [ ]:
agent.epsilon

In [ ]:
plt.figure(figsize=(10, 6))
x = range(len(agent.averages))
y = np.polyval(np.polyfit(x, agent.averages, deg=3), x)
plt.plot(agent.averages, label='total rewards (average)')
plt.plot(x, y, 'r--')
plt.xlabel('episodes')
plt.ylabel('total reward')
plt.legend();

In [ ]:
plt.figure(figsize=(10, 6))
x = range(len(agent.aperformances))
y = np.polyval(np.polyfit(x, agent.aperformances, deg=3), x)
plt.plot(x[25:], agent.aperformances[25:], label='performance')
plt.plot(x[25:], y[25:], 'r--')
plt.xlabel('episodes')
plt.ylabel('total reward')
plt.legend();

In [ ]:
trewards, performances = agent.test(3, max_iter=2600)

In [ ]:
performances

<img src='https://hilpisch.com/taim_logo.png' width="350px" align="right">